**SEQUENCE TO SEQUENCE (seq2seq)**

Architect an encoder-decoder network in Pytorch for machine translation without using pre-built Transformer modules.

In [3]:
import random
import torch
import torch.nn as nn
import torch.optim as optim

In [4]:
#REPRODUCIBILITY

random.seed(42)
torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: cuda


In [5]:
#VOCABULARY HELPER CLASS

class Vocabulary:

    def __init__(self, name):

        self.name = name

        # Special tokens
        self.word2index = {
            "<PAD>": 0,
            "<SOS>": 1,
            "<EOS>": 2,
            "<UNK>": 3
        }

        self.index2word = {
            0: "<PAD>",
            1: "<SOS>",
            2: "<EOS>",
            3: "<UNK>"
        }

        self.num_words = 4


    def add_sentence(self, sentence):

        for word in sentence.lower().split():

            if word not in self.word2index:

                self.word2index[word] = self.num_words
                self.index2word[self.num_words] = word

                self.num_words += 1


    def numericalize(self, sentence):

        return [
            self.word2index.get(
                word,
                self.word2index["<UNK>"]
            )
            for word in sentence.lower().split()
        ]


In [6]:
#ENCODER
class Encoder(nn.Module):

    def __init__(
        self,
        input_dim,
        emb_dim,
        hidden_dim
    ):

        super().__init__()

        # Convert word IDs → dense vectors
        self.embedding = nn.Embedding(
            input_dim,
            emb_dim
        )

        # Process the source sequence
        self.lstm = nn.LSTM(
            emb_dim,
            hidden_dim,
            batch_first=True
        )


    def forward(self, src):

        # src:
        # [batch_size, source_sequence_length]

        embedded = self.embedding(src)

        # embedded:
        # [batch_size, source_sequence_length, embedding_dim]

        outputs, (hidden, cell) = self.lstm(
            embedded
        )

        # hidden:
        # [1, batch_size, hidden_dim]

        # cell:
        # [1, batch_size, hidden_dim]

        return hidden, cell

In [7]:
#DECODER

class Decoder(nn.Module):

    def __init__(
        self,
        output_dim,
        emb_dim,
        hidden_dim
    ):

        super().__init__()

        # Convert target token IDs → embeddings
        self.embedding = nn.Embedding(
            output_dim,
            emb_dim
        )

        # Generate target sequence
        self.lstm = nn.LSTM(
            emb_dim,
            hidden_dim,
            batch_first=True
        )

        # Convert hidden state → vocabulary scores
        self.fc = nn.Linear(
            hidden_dim,
            output_dim
        )


    def forward(
        self,
        input_token,
        hidden,
        cell
    ):

        # input_token:
        # [batch_size, 1]

        embedded = self.embedding(
            input_token
        )

        # embedded:
        # [batch_size, 1, embedding_dim]

        output, (hidden, cell) = self.lstm(
            embedded,
            (hidden, cell)
        )

        # output:
        # [batch_size, 1, hidden_dim]

        prediction = self.fc(output)

        # prediction:
        # [batch_size, 1, target_vocab_size]

        return prediction, hidden, cell

In [9]:
#SEQ2SEQ MODEL

class Seq2Seq(nn.Module):

    def __init__(
        self,
        encoder,
        decoder
    ):

        super().__init__()

        self.encoder = encoder
        self.decoder = decoder


    def forward(
        self,
        src,
        trg,
        teacher_forcing_ratio=0.5
    ):

        batch_size = trg.shape[0]
        trg_len = trg.shape[1]

        output_vocab_size = (
            self.decoder.fc.out_features
        )

        # Store decoder predictions
        outputs = torch.zeros(
            batch_size,
            trg_len,
            output_vocab_size,
            device=src.device
        )

        # --------------------------------------------------------
        # ENCODER
        # --------------------------------------------------------

        hidden, cell = self.encoder(src)

        # First decoder input = <SOS>
        input_token = trg[:, 0].unsqueeze(1)

        # --------------------------------------------------------
        # DECODER
        # --------------------------------------------------------

        for t in range(1, trg_len):

            output, hidden, cell = self.decoder(
                input_token,
                hidden,
                cell
            )

            # Save prediction
            outputs[:, t] = output.squeeze(1)

            # Decide whether to use:
            # actual target OR model prediction
            teacher_force = (
                random.random()
                < teacher_forcing_ratio
            )

            # Best predicted token
            top1 = output.argmax(2)

            if teacher_force:

                input_token = (
                    trg[:, t].unsqueeze(1)
                )

            else:

                input_token = top1

        return outputs

In [10]:
#small dataset made from fra.txt (manythings data)

raw_lines = [

    "Go.\tVa !\tCC-BY 2.0 (France) Attribution",

    "Hi.\tSalut.\tCC-BY 2.0 (France) Attribution",

    "Run!\tCours !\tCC-BY 2.0 (France) Attribution",

    "I know.\tJe sais.\tCC-BY 2.0 (France) Attribution",

    "Try it.\tEssaye-le.\tCC-BY 2.0 (France) Attribution",

    "I'm glad.\tJe suis ravi.\tCC-BY 2.0 (France) Attribution"

]

In [12]:
#use this if you have enough memory to run


"""
with open(
     "/content/fra.txt",
     "r",
     encoding="utf-8"
 ) as f:

     raw_lines = f.readlines()
"""



'\nwith open(\n     "/content/fra.txt",\n     "r",\n     encoding="utf-8"\n ) as f:\n\n     raw_lines = f.readlines()\n'

In [13]:
#BUILD VOCABULARIES
src_vocab = Vocabulary("English")
trg_vocab = Vocabulary("French")

pairs = []


for line in raw_lines:

    if line.strip():

        parts = line.split("\t")

        if len(parts) >= 2:

            english_sentence = parts[0].strip()
            french_sentence = parts[1].strip()

            # Add words to vocabularies
            src_vocab.add_sentence(
                english_sentence
            )

            trg_vocab.add_sentence(
                french_sentence
            )

            pairs.append(
                (
                    english_sentence,
                    french_sentence
                )
            )


In [14]:
#MODEL HYPERPARAMETERS

INPUT_DIM = src_vocab.num_words
OUTPUT_DIM = trg_vocab.num_words

EMBED_DIM = 64
HIDDEN_DIM = 128

#Recommended learning rate
LEARNING_RATE = 0.001

#More epochs because the demonstration dataset contains only six sentence pairs
EPOCHS = 500

In [15]:
#CREATE MODEL

encoder = Encoder(
    INPUT_DIM,
    EMBED_DIM,
    HIDDEN_DIM
)

decoder = Decoder(
    OUTPUT_DIM,
    EMBED_DIM,
    HIDDEN_DIM
)

model = Seq2Seq(
    encoder,
    decoder
).to(device)


In [16]:
#LOSS FUNCTION & OPTIMIZER

# Ignore PAD tokens while calculating loss
criterion = nn.CrossEntropyLoss(
    ignore_index=src_vocab.word2index["<PAD>"]
)

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)


In [17]:
#CONVERT SENTENCES INTO TENSORS

src_list = []
trg_list = []


# Find maximum sequence lengths
max_src_len = max(
    len(sentence.split())
    for sentence, _ in pairs
) + 2


max_trg_len = max(
    len(sentence.split())
    for _, sentence in pairs
) + 2


for english_sentence, french_sentence in pairs:

    # ------------------------------------------------------------
    # SOURCE
    # ------------------------------------------------------------

    src_tokens = [

        src_vocab.word2index["<SOS>"]

    ]

    src_tokens += src_vocab.numericalize(
        english_sentence
    )

    src_tokens.append(
        src_vocab.word2index["<EOS>"]
    )


    # ------------------------------------------------------------
    # TARGET
    # ------------------------------------------------------------

    trg_tokens = [

        trg_vocab.word2index["<SOS>"]

    ]

    trg_tokens += trg_vocab.numericalize(
        french_sentence
    )

    trg_tokens.append(
        trg_vocab.word2index["<EOS>"]
    )


    # ------------------------------------------------------------
    # PADDING
    # ------------------------------------------------------------

    src_tokens += [

        src_vocab.word2index["<PAD>"]

    ] * (
        max_src_len - len(src_tokens)
    )


    trg_tokens += [

        trg_vocab.word2index["<PAD>"]

    ] * (
        max_trg_len - len(trg_tokens)
    )


    src_list.append(src_tokens)
    trg_list.append(trg_tokens)


# Convert lists → tensors
src_tensor = torch.tensor(
    src_list,
    dtype=torch.long
).to(device)


trg_tensor = torch.tensor(
    trg_list,
    dtype=torch.long
).to(device)

In [18]:
#DISPLAY DATASET INFORMATION
print("\n" + "=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print(
    f"Number of sentence pairs : {len(pairs)}"
)

print(
    f"English vocabulary size  : {INPUT_DIM}"
)

print(
    f"French vocabulary size   : {OUTPUT_DIM}"
)

print(
    f"Maximum source length    : {max_src_len}"
)

print(
    f"Maximum target length    : {max_trg_len}"
)

print(
    f"Embedding dimension      : {EMBED_DIM}"
)

print(
    f"Hidden dimension         : {HIDDEN_DIM}"
)

print(
    f"Learning rate            : {LEARNING_RATE}"
)

print(
    f"Training epochs          : {EPOCHS}"
)


DATASET INFORMATION
Number of sentence pairs : 6
English vocabulary size  : 13
French vocabulary size   : 13
Maximum source length    : 4
Maximum target length    : 5
Embedding dimension      : 64
Hidden dimension         : 128
Learning rate            : 0.001
Training epochs          : 500


In [20]:
#TRAINING LOOP
print("TRAINING")



model.train()


for epoch in range(1, EPOCHS + 1):

    # Clear old gradients
    optimizer.zero_grad()


    # ------------------------------------------------------------
    # FORWARD PASS
    # ------------------------------------------------------------

    output = model(
        src_tensor,
        trg_tensor,
        teacher_forcing_ratio=0.6
    )


    # output:
    # [batch_size, target_length, output_vocab_size]


    # Remove <SOS> position
    output = output[:, 1:]


    # Flatten predictions
    output = output.reshape(
        -1,
        OUTPUT_DIM
    )


    # Remove <SOS> from target
    targets = trg_tensor[:, 1:]


    # Flatten targets
    targets = targets.reshape(-1)


    # ------------------------------------------------------------
    # LOSS
    # ------------------------------------------------------------

    loss = criterion(
        output,
        targets
    )


    # ------------------------------------------------------------
    # BACKPROPAGATION
    # ------------------------------------------------------------

    loss.backward()


    # ------------------------------------------------------------
    # UPDATE PARAMETERS
    # ------------------------------------------------------------

    optimizer.step()


    # ------------------------------------------------------------
    # DISPLAY LOSS
    # ------------------------------------------------------------

    if (
        epoch == 1
        or epoch % 50 == 0
    ):

        print(
            f"Epoch: {epoch:03d} | "
            f"Loss: {loss.item():.4f}"
        )


TRAINING
Epoch: 001 | Loss: 0.0010
Epoch: 050 | Loss: 0.0009
Epoch: 100 | Loss: 0.0008
Epoch: 150 | Loss: 0.0007
Epoch: 200 | Loss: 0.0006
Epoch: 250 | Loss: 0.0005
Epoch: 300 | Loss: 0.0005
Epoch: 350 | Loss: 0.0004
Epoch: 400 | Loss: 0.0004
Epoch: 450 | Loss: 0.0003
Epoch: 500 | Loss: 0.0003


In [24]:
#STANDALONE TRANSLATION FUNCTION

# This function does NOT require the target French sentence.

# It demonstrates actual autoregressive decoding:

# English → Encoder → <SOS> → prediction → prediction → ... → <EOS>


def translate_sentence(
    model,
    sentence,
    src_vocab,
    trg_vocab,
    max_len=20
):

    model.eval()


    # ============================================================
    # STEP 1 — Convert input sentence to token IDs
    # ============================================================

    src_tokens = [

        src_vocab.word2index["<SOS>"]

    ]

    src_tokens += src_vocab.numericalize(
        sentence
    )

    src_tokens.append(
        src_vocab.word2index["<EOS>"]
    )


    # Convert to tensor
    src = torch.tensor(
        src_tokens,
        dtype=torch.long
    ).unsqueeze(0).to(device)


    # ============================================================
    # STEP 2 — Encode source sentence
    # ============================================================

    with torch.no_grad():

        hidden, cell = model.encoder(
            src
        )


    # ============================================================
    # STEP 3 — Start decoder with <SOS>
    # ============================================================

    input_token = torch.tensor(

        [[trg_vocab.word2index["<SOS>"]]],

        dtype=torch.long

    ).to(device)


    predicted_words = []


    # ============================================================
    # STEP 4 — AUTOREGRESSIVE DECODING
    # ============================================================

    for _ in range(max_len):

        with torch.no_grad():

            output, hidden, cell = model.decoder(
                input_token,
                hidden,
                cell
            )


        # Select highest probability token
        prediction = output.argmax(2)


        predicted_id = prediction.item()


        # ========================================================
        # Stop when <EOS> is generated
        # ========================================================

        if predicted_id == trg_vocab.word2index["<EOS>"]:

            break


        # Ignore special tokens
        if predicted_id not in [

            trg_vocab.word2index["<PAD>"],

            trg_vocab.word2index["<SOS>"]

        ]:

            predicted_words.append(

                trg_vocab.index2word[
                    predicted_id
                ]

            )


        # ========================================================
        # Feed prediction back into decoder
        # ========================================================

        input_token = prediction


    return " ".join(
        predicted_words
    )

In [26]:
#INFERENCE / TESTING
print("STANDALONE TRANSLATION")



test_sentences = [

    "Go.",

    "Hi.",

    "Run!",

    "I know.",

    "Try it.",

    "I'm glad."

]


for sentence in test_sentences:

    prediction = translate_sentence(

        model,

        sentence,

        src_vocab,

        trg_vocab

    )


    print(
        f"English   : {sentence}"
    )

    print(
        f"Predicted : {prediction}"
    )

    print("-" * 60)

STANDALONE TRANSLATION
English   : Go.
Predicted : va !
------------------------------------------------------------
English   : Hi.
Predicted : salut.
------------------------------------------------------------
English   : Run!
Predicted : cours !
------------------------------------------------------------
English   : I know.
Predicted : je sais.
------------------------------------------------------------
English   : Try it.
Predicted : essaye-le.
------------------------------------------------------------
English   : I'm glad.
Predicted : je suis ravi.
------------------------------------------------------------


In [25]:
#TRAINING DATA COMPARISON


print("\n" + "=" * 60)
print("TARGET vs PREDICTED")
print("=" * 60)


for english, target in pairs:

    prediction = translate_sentence(

        model,

        english,

        src_vocab,

        trg_vocab

    )


    print(
        f"English   : {english}"
    )

    print(
        f"Target    : {target}"
    )

    print(
        f"Predicted : {prediction}"
    )

    print("-" * 60)


TARGET vs PREDICTED
English   : Go.
Target    : Va !
Predicted : va !
------------------------------------------------------------
English   : Hi.
Target    : Salut.
Predicted : salut.
------------------------------------------------------------
English   : Run!
Target    : Cours !
Predicted : cours !
------------------------------------------------------------
English   : I know.
Target    : Je sais.
Predicted : je sais.
------------------------------------------------------------
English   : Try it.
Target    : Essaye-le.
Predicted : essaye-le.
------------------------------------------------------------
English   : I'm glad.
Target    : Je suis ravi.
Predicted : je suis ravi.
------------------------------------------------------------
